In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [4]:
### FLOW MATCHING!! ###

In [5]:
### PROBABILISTIC GENERATIVE AI APPROACH!! ###

In [9]:
# specify new dataloader that adds different "noise" amounts to y-values
# as these methods work with noisy data

from torch.utils.data import Dataset, DataLoader
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class FlowMatchingDataset(Dataset):
    def __init__(self, data_x, data_y, n_samples=1000, sigma_min=1e-4):
        super().__init__()
        self.n_samples = n_samples
        self.sigma_min = sigma_min
        self.data_x = data_x
        self.data_y = data_y

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x0 = np.random.multivariate_normal([0.0, 0.0], np.eye(2), 1)[0]
        t = np.random.rand() # scalar in [0,1]
        dx = self.data_x[idx] #:idx+1]
        dy_org = self.data_y[idx] #:idx+1]
        x0[0] = dx[0] # keep x value
        x1 = np.concatenate([dx,dy_org],axis=0)
        # print([self.data_x.shape, dx.shape, x1.shape])

        x_t = x0*(1 - (1 - self.sigma_min)*t) + x1*t
        u_t = (x1-x0)
        x_t = torch.tensor(x_t, dtype=torch.float32)
        t = torch.tensor([t], dtype=torch.float32)
        u_t = torch.tensor(u_t, dtype=torch.float32)
        return x_t, t, u_t

In [10]:
# add time input t to network similar to before

class VelocityNet(nn.Module):
    def __init__(self, hiddendim, in_dim=2, time_dim=1, out_dim=2):
        super().__init__()
        self.net == nn.sequential(
            nn.Linear(in_dim + time_dim, hiddendim),
            nn.ReLU(),
            nn.Linear(hiddendim, hiddendim),
            nn.ReLU(),
            nn.Linear(hiddendim, out_dim)
        )

    def forward(self, x, t):
        xt = torch.cat([x, t], dim=1)
        return self.net(xt)

In [11]:
## same X and Y as before

N = 10000
X = np.random.random(N).astype(np.float32).reshape(-1,1)

# generate Y-data
sign = (- np.ones((N,))).astype(np.float32) ** np.random.randint(2, size=N)
Y = (np.sqrt(X.flatten()) * sign).reshape(-1,1).astype(np.float32)

In [ ]:
# run sampled from noisy dataset, similar otherwise

batch_size = 128
epochs = 50

dataset = FlowMatchingDataset(X, Y, n_samples=N)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

nn_fm = VelocityNet(hiddendim = 128).to(device)
optimizer = optim.Adam(nn_fm.parameters(), lr=0.001)
criterion = nn.MSELoss()

for epoch in range(epochs):
    